In [6]:
import requests
import pandas as pd
import re
from datetime import datetime

client_id = "tUpceN2tI9JBcwm05yBW"
client_secret = "OW_SC2bJMD"

url = "https://openapi.naver.com/v1/search/news.json"

headers = {
    "X-Naver-Client-Id": client_id,
    "X-Naver-Client-Secret": client_secret
}

keywords = [
    "숏폼",
    "쇼츠",
    "릴스",
    "틱톡",
    "숏폼드라마"
]

def clean_text(text):
    if not text:
        return ""
    
    text = re.sub("<.*?>", "", text)
    text = text.replace("&quot;", '"')
    text = text.replace("&amp;", "&")
    text = text.replace("&lt;", "<")
    text = text.replace("&gt;", ">")
    return text.strip()

rows = []

for keyword in keywords:
    params = {
        "query": keyword,
        "display": 100,
        "start": 1,
        "sort": "date"
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print("API 요청 실패:", keyword)
        print(response.status_code)
        print(response.text)
        continue

    data = response.json()

    for item in data.get("items", []):
        rows.append({
            "keyword": keyword,
            "title": clean_text(item.get("title", "")),
            "description": clean_text(item.get("description", "")),
            "originallink": item.get("originallink", ""),
            "link": item.get("link", ""),
            "pub_date": item.get("pubDate", ""),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

df = pd.DataFrame(rows)

df.insert(0, "index", range(1, len(df) + 1))

df.to_csv(
    "naver_news_shortform.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.head())
print("뉴스 데이터 저장 완료")

   index keyword                                           title  \
0      1      숏폼     이준, 근육질 걸그룹 댄스에 유재석 폭소..'캐치 캐치' 이어 아일릿도 ...   
1      2      숏폼  [K-컬처가 여는 스마트관광의 미래 (6)] K-콘텐츠의 힘, 영화·드라마·팝...   
2      3      숏폼        [이대화의 함께 들어요] [40] 빨라진 K팝 노래, 배경엔 숏폼이 있다   
3      4      숏폼            "생명의 길 사계절 담으세요"… 농어촌공사, 어도 사진 공모 예고   
4      5      숏폼                 농협상호금융, 대학생 홍보단 ‘NH콕서포터즈’ 5기 모집   

                                         description  \
0  앞서 이준은 유튜브 '워크맨'에서 치어리더로 변신해 가수 최예나의 '캐치캐치' 댄스...   
1  틱톡 숏폼 영상, 유튜브 브이로그, 인스타그램 릴스와 같은 플랫폼이 관광객을 또 다...   
2  요즘 유행의 키를 쥔 숏폼 플랫폼이 속도 높은 음악들을 선호하면서 자연스럽게 흐름이...   
3  올해 공모전은 사진과 짧은 영상(숏폼) 등 2개 부문으로 나눠 운영된다. 사진 부문...   
4  카드뉴스와 숏폼 영상, SNS 콘텐츠 등을 직접 기획·제작하며 디지털 금융 서비스를...   

                                        originallink  \
0  https://www.starnewskorea.com/broadcast-show/2...   
1  https://www.news2day.co.kr/article/20260513500162   
2  https://www.chosun.com/opinion/specialist_colu...   
3  https://www.newscj.com/news